In [1]:
# Ensure jacopy is importable when this notebook is opened
# directly (not via pytest). Walks up from the notebook's
# directory to the repo root and prepends it to sys.path if
# jacopy isn't already installed into this kernel.
try:
    import jacopy  # noqa: F401
except ModuleNotFoundError:
    import sys
    from pathlib import Path
    here = Path.cwd().resolve()
    for candidate in (here, *here.parents):
        if (candidate / "jacopy" / "__init__.py").is_file():
            sys.path.insert(0, str(candidate))
            break
    import jacopy  # noqa: F401

# 08, The unified picture

Companion notebook to [08_unified_picture.md](08_unified_picture.md). A single hypothesis `[π, π]_SN = 0`, same obstruction at both the function and form levels. Theorem Book citation chain. A parallel: `dH = 0`.

## One hypothesis, two faces

`jacobi_condition` and `koszul_jacobi_condition` point to the same `Expr`, only the display name differs.

In [2]:
from jacopy.library.declarations import Bivector, Forms, Functions
from jacopy.library.poisson import PoissonBracket
from jacopy.core.registry import PropertyRegistry

reg = PropertyRegistry()
pi = Bivector('π', registry=reg)
poisson = PoissonBracket.from_bivector(pi)

c1 = poisson.jacobi_condition(reg)
c2 = poisson.koszul_jacobi_condition(reg)
print('func obs:', c1.obstruction)
print('form obs:', c2.obstruction)
print('same expr?:', c1.obstruction == c2.obstruction)

func obs: [·,·]_SN(π, π)
form obs: [·,·]_SN(π, π)
same expr?: True


## Function vs form, meeting at the same obstruction

In [3]:
f, g, h = Functions('f g h', degree=-1, registry=reg)
alpha, beta, gamma = Forms('α β γ', degree=1, registry=reg)

func_chain = poisson.prove_jacobi_reduction(f, g, h, registry=reg)
form_chain = poisson.prove_koszul_jacobi_reduction(
    alpha, beta, gamma, registry=reg
)

print('func rule:', func_chain.steps[0].rule)
print('form rule:', form_chain.steps[0].rule)
print('func after:', func_chain.steps[0].after)
print('form after:', form_chain.steps[0].after)
print('same after?:', func_chain.steps[0].after == form_chain.steps[0].after)

func rule: DerivedBracketTheorem
form rule: DerivedBracketTheorem
func after: [·,·]_SN(π, π)
form after: [·,·]_SN(π, π)
same after?: True


## The classical–derived bridge, the `reflexive` step

The package's expand rules bring both sides to the same Expr tree; Koszul equivalence closes structurally.

In [4]:
chain_eq = poisson.prove_koszul_equivalence(alpha, beta, registry=reg)
print('len:', len(chain_eq), 'rule:', chain_eq.steps[0].rule)

len: 1 rule: reflexive


## Seeded theorems, the citation chain

In [5]:
from jacopy.library import theorem_book

for name in (
    'poisson_jacobi',
    'poisson_koszul_equivalence',
    'poisson_koszul_jacobi',
):
    thm = theorem_book.get(name)
    print(name)
    for ax in thm.from_axioms:
        print('   -', ax)

poisson_jacobi
   - Derived Bracket Theorem
   - [π, π]_SN = 0 (Poisson hypothesis)
poisson_koszul_equivalence
   - derived bracket definition
   - classical Koszul bracket definition
   - π^♯ = Sharp(π) as common anchor
poisson_koszul_jacobi
   - Derived Bracket Theorem
   - π^♯ = Sharp(π) as form-lift anchor
   - [π, π]_SN = 0 (Poisson hypothesis)


## `dH = 0`, the same pattern on the Courant side

In [6]:
from jacopy.brackets.courant import CourantBracket
from jacopy.core.expr import Symbol
from jacopy.core.properties import Graded

reg_h = PropertyRegistry()
H = Symbol('H')
reg_h.declare(H, Graded(degree=3))

C = CourantBracket(background_H=H)
cond = C.jacobi_condition(reg_h)
print('name       :', cond.name)
print('obstruction:', cond.obstruction)

twist = theorem_book.get('courant_jacobi_twist')
print('twist from_axioms:', twist.from_axioms)

name       : Courant Jacobi condition (H-twisted by H)
obstruction: d(H)
twist from_axioms: ('Courant algebroid Jacobi axiom', 'dH = 0 (closed-3-form hypothesis)')


## Next step

Where does the axiom *itself* come from? [09_foundations.md](09_foundations.md).